In [1]:
import os
import pandas as pd
from rdkit import Chem

# 파일 정보 (경로 + 측정 타입 컬럼명)
file_infos = {
    "ChEMBL": {
        "path": " /summer_conference/open/ChEMBL_ASK1(IC50)_clean.csv",
        "type_col": "Standard Type"
    },
    "CAS": {
        "path": " /summer_conference/open/CAS_clean.csv",
        "type_col": "Assay Parameter"
    },
    "PubChem": {
        "path": " /summer_conference/open/Pubchem_clean.csv",
        "type_col": "Activity_Type"
    }
}

# 결과 저장
filtered_dfs = []

# 안전한 Mol 변환 함수
def safe_mol_from_smiles(smiles):
    if isinstance(smiles, str) and smiles.strip():
        try:
            return Chem.MolFromSmiles(smiles)
        except:
            return None
    return None

# 각 데이터셋 처리
for source_name, info in file_infos.items():
    path = info["path"]
    type_col = info["type_col"]

    if not os.path.exists(path):
        raise FileNotFoundError(f"{source_name}: ❌ 파일이 존재하지 않습니다: {path}")

    # 데이터 로드
    df = pd.read_csv(path, encoding="cp1252")
    print(f"\n📁 {source_name} 로드 완료: {df.shape[0]}행")

    # pIC50 컬럼명 자동 탐지
    if 'pIC50' in df.columns:
        pic50_col = 'pIC50'
    elif 'pX Value' in df.columns:
        pic50_col = 'pX Value'
    elif 'Assay Parameter Value' in df.columns:
        pic50_col = 'Assay Parameter Value'
    elif 'Single Value (Parsed)' in df.columns:
        pic50_col = 'Single Value (Parsed)'
    else:
        raise ValueError(f"{source_name}: pIC50 값 컬럼을 찾을 수 없습니다.")

    # IC50 필터링
    if type_col not in df.columns:
        raise ValueError(f"{source_name}: 측정 타입 컬럼 '{type_col}'이 존재하지 않습니다.")
    df_ic50 = df[df[type_col].astype(str).str.upper() == 'IC50']

    # 필수 컬럼 확인
    if 'SMILES' not in df_ic50.columns:
        raise KeyError(f"{source_name}: 'SMILES' 컬럼이 없습니다.")
    
    # SMILES → mol
    df_ic50['mol'] = df_ic50['SMILES'].apply(safe_mol_from_smiles)
    df_ic50 = df_ic50.dropna(subset=['mol'])  # 유효 mol만

    # 컬럼 정리
    df_ic50 = df_ic50[['SMILES', pic50_col, 'mol']].copy()
    df_ic50.rename(columns={pic50_col: 'pIC50'}, inplace=True)
    df_ic50['Source'] = source_name

    print(f"✅ {source_name} IC50 + mol 유효 데이터 수: {df_ic50.shape[0]}")

    filtered_dfs.append(df_ic50)

# 병합
combined_df = pd.concat(filtered_dfs, ignore_index=True)
print(f"\n📊 총 유효 IC50 mol 데이터 수: {combined_df.shape[0]}")

# (선택) 저장
# combined_df.to_csv("merged_IC50_cleaned_with_mol.csv", index=False)



📁 ChEMBL 로드 완료: 824행
✅ ChEMBL IC50 + mol 유효 데이터 수: 824

📁 CAS 로드 완료: 3452행


/var/folders/2q/v_79vqld72sf48ghq2z01j2m0000gn/T/ipykernel_55072/3304300268.py:67: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_ic50['mol'] = df_ic50['SMILES'].apply(safe_mol_from_smiles)


✅ CAS IC50 + mol 유효 데이터 수: 2844

📁 PubChem 로드 완료: 23795행
✅ PubChem IC50 + mol 유효 데이터 수: 1039

📊 총 유효 IC50 mol 데이터 수: 4707


/var/folders/2q/v_79vqld72sf48ghq2z01j2m0000gn/T/ipykernel_55072/3304300268.py:67: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_ic50['mol'] = df_ic50['SMILES'].apply(safe_mol_from_smiles)


In [2]:
from rdkit import Chem
from rdkit.Chem import Descriptors
import pandas as pd

# 예: combined_df에는 'SMILES', 'pIC50', 'mol', 'Source' 컬럼이 있다고 가정

def featurize_minimal(mol):
    return {
        'Num_N': sum(atom.GetSymbol() == 'N' for atom in mol.GetAtoms()),
        'TPSA': Descriptors.TPSA(mol)
    }

# mol 피처 추출
feats = [featurize_minimal(mol) for mol in combined_df['mol']]

# 피처 DataFrame 생성
feat_df = pd.DataFrame(feats)

# 기존 데이터와 병합
df_final = pd.concat([combined_df[['SMILES', 'pIC50', 'Source']], feat_df], axis=1)

print(df_final.head())
print(f"최종 데이터 크기: {df_final.shape}")


                                              SMILES     pIC50  Source  Num_N  \
0   Cn1cc(Cl)c2cnc(NC(=O)c3ccc([C@](C)(O)CO)cc3)cc21  7.420216  ChEMBL      3   
1         Cc1cc2c(-c3ccc(S(=O)(=O)NCCN)s3)ccnc2[nH]1  6.600326  ChEMBL      4   
2  Cc1cc(C)c2nc(N3C(=O)C(O)=C(C(=O)c4ccc(Cl)cc4)C...  5.200659  ChEMBL      2   
3  Cc1ccc(C(=O)C2=C(O)C(=O)N(c3nc4ccc(F)cc4s3)C2c...  5.119186  ChEMBL      2   
4  CCOc1ccc2nc(N3C(=O)C(O)=C(C(=O)c4ccccc4)C3c3cc...  5.376751  ChEMBL      2   

     TPSA  
0   87.38  
1  100.87  
2   70.50  
3   70.50  
4   79.73  
최종 데이터 크기: (4707, 5)


In [3]:
from rdkit.Chem import AllChem
import numpy as np
import pandas as pd

# Morgan Fingerprint 계산 함수 (radius=2, 길이=2048 비트)
def mol_to_morgan_fp(mol, radius=2, n_bits=2048):
    if mol is None:
        return np.zeros(n_bits, dtype=int)
    fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius, nBits=n_bits)
    arr = np.zeros((n_bits,), dtype=int)
    AllChem.DataStructs.ConvertToNumpyArray(fp, arr)
    return arr

# combined_df에 mol 컬럼이 있다고 가정
fp_array = np.array([mol_to_morgan_fp(mol) for mol in combined_df['mol']])

# numpy 배열을 데이터프레임으로 변환
fp_df = pd.DataFrame(fp_array, columns=[f'FP_{i}' for i in range(fp_array.shape[1])])

# 원본 데이터와 병합
df_with_fp = pd.concat([combined_df.reset_index(drop=True), fp_df], axis=1)

print(df_with_fp.head())
print(f"최종 데이터 크기 (피처 포함): {df_with_fp.shape}")


[17:44:18] DEPRECATION WARNING: please use MorganGenerator
[17:44:18] DEPRECATION WARNING: please use MorganGenerator
[17:44:18] DEPRECATION WARNING: please use MorganGenerator
[17:44:18] DEPRECATION WARNING: please use MorganGenerator
[17:44:18] DEPRECATION WARNING: please use MorganGenerator
[17:44:18] DEPRECATION WARNING: please use MorganGenerator
[17:44:18] DEPRECATION WARNING: please use MorganGenerator
[17:44:18] DEPRECATION WARNING: please use MorganGenerator
[17:44:18] DEPRECATION WARNING: please use MorganGenerator
[17:44:18] DEPRECATION WARNING: please use MorganGenerator
[17:44:18] DEPRECATION WARNING: please use MorganGenerator
[17:44:18] DEPRECATION WARNING: please use MorganGenerator
[17:44:18] DEPRECATION WARNING: please use MorganGenerator
[17:44:18] DEPRECATION WARNING: please use MorganGenerator
[17:44:18] DEPRECATION WARNING: please use MorganGenerator
[17:44:18] DEPRECATION WARNING: please use MorganGenerator
[17:44:18] DEPRECATION WARNING: please use MorganGenerat

                                              SMILES     pIC50  \
0   Cn1cc(Cl)c2cnc(NC(=O)c3ccc([C@](C)(O)CO)cc3)cc21  7.420216   
1         Cc1cc2c(-c3ccc(S(=O)(=O)NCCN)s3)ccnc2[nH]1  6.600326   
2  Cc1cc(C)c2nc(N3C(=O)C(O)=C(C(=O)c4ccc(Cl)cc4)C...  5.200659   
3  Cc1ccc(C(=O)C2=C(O)C(=O)N(c3nc4ccc(F)cc4s3)C2c...  5.119186   
4  CCOc1ccc2nc(N3C(=O)C(O)=C(C(=O)c4ccccc4)C3c3cc...  5.376751   

                                             mol  Source  FP_0  FP_1  FP_2  \
0  <rdkit.Chem.rdchem.Mol object at 0x16f3053c0>  ChEMBL     0     0     0   
1  <rdkit.Chem.rdchem.Mol object at 0x16f305430>  ChEMBL     0     0     0   
2  <rdkit.Chem.rdchem.Mol object at 0x16f3054a0>  ChEMBL     0     1     0   
3  <rdkit.Chem.rdchem.Mol object at 0x16f305510>  ChEMBL     0     0     0   
4  <rdkit.Chem.rdchem.Mol object at 0x16f305580>  ChEMBL     0     0     0   

   FP_3  FP_4  FP_5  ...  FP_2038  FP_2039  FP_2040  FP_2041  FP_2042  \
0     0     0     0  ...        0        0        0        0 

In [4]:
from rdkit.Chem import Descriptors

def calc_extra_features(mol):
    if mol is None:
        return {'Num_N': 0, 'TPSA': 0.0}
    num_n = sum(atom.GetSymbol() == 'N' for atom in mol.GetAtoms())
    tpsa = Descriptors.TPSA(mol)
    return {'Num_N': num_n, 'TPSA': tpsa}

# extra features 계산
extra_feats = [calc_extra_features(mol) for mol in combined_df['mol']]
extra_feat_df = pd.DataFrame(extra_feats)

# 기존 fingerprint와 합치기
df_with_extra = pd.concat([df_with_fp.reset_index(drop=True), extra_feat_df], axis=1)

print(df_with_extra.head())
print(f"최종 데이터 크기 (피처 + Num_N, TPSA 포함): {df_with_extra.shape}")


                                              SMILES     pIC50  \
0   Cn1cc(Cl)c2cnc(NC(=O)c3ccc([C@](C)(O)CO)cc3)cc21  7.420216   
1         Cc1cc2c(-c3ccc(S(=O)(=O)NCCN)s3)ccnc2[nH]1  6.600326   
2  Cc1cc(C)c2nc(N3C(=O)C(O)=C(C(=O)c4ccc(Cl)cc4)C...  5.200659   
3  Cc1ccc(C(=O)C2=C(O)C(=O)N(c3nc4ccc(F)cc4s3)C2c...  5.119186   
4  CCOc1ccc2nc(N3C(=O)C(O)=C(C(=O)c4ccccc4)C3c3cc...  5.376751   

                                             mol  Source  FP_0  FP_1  FP_2  \
0  <rdkit.Chem.rdchem.Mol object at 0x16f3053c0>  ChEMBL     0     0     0   
1  <rdkit.Chem.rdchem.Mol object at 0x16f305430>  ChEMBL     0     0     0   
2  <rdkit.Chem.rdchem.Mol object at 0x16f3054a0>  ChEMBL     0     1     0   
3  <rdkit.Chem.rdchem.Mol object at 0x16f305510>  ChEMBL     0     0     0   
4  <rdkit.Chem.rdchem.Mol object at 0x16f305580>  ChEMBL     0     0     0   

   FP_3  FP_4  FP_5  ...  FP_2040  FP_2041  FP_2042  FP_2043  FP_2044  \
0     0     0     0  ...        0        0        0        0 

In [5]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

def train_and_evaluate(model_name, X, y, test_size=0.2, random_state=42):
    # 1) 필요없는 컬럼 제거 (이미 df_with_extra 기준)
    X = X.drop(columns=['SMILES', 'pIC50', 'mol', 'Source'], errors='ignore')
    
    # 2) 결측치 제거
    valid_idx = y.notnull() & X.notnull().all(axis=1)
    X_valid = X[valid_idx]
    y_valid = y[valid_idx]

    # 3) 데이터 분할
    X_train, X_test, y_train, y_test = train_test_split(
        X_valid, y_valid, test_size=test_size, random_state=random_state)

    # 4) 모델 선택
    if model_name == 'RandomForest':
        model = RandomForestRegressor(n_estimators=100, random_state=random_state)
    elif model_name == 'XGBoost':
        model = XGBRegressor(n_estimators=100, random_state=random_state, verbosity=0)
    elif model_name == 'LightGBM':
        model = LGBMRegressor(n_estimators=100, random_state=random_state)
    else:
        raise ValueError(f"지원하지 않는 모델 이름입니다: {model_name}")

    # 5) 학습
    model.fit(X_train, y_train)

    # 6) 예측 및 평가
    y_pred = model.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)

    print(f"== {model_name} 결과 ==")
    print(f"RMSE: {rmse:.3f}")
    print(f"R^2: {r2:.3f}")
    print()

    return model, rmse, r2

# 사용 예시
model_rf, rmse_rf, r2_rf = train_and_evaluate('RandomForest', df_with_extra, df_with_extra['pIC50'])
model_xgb, rmse_xgb, r2_xgb = train_and_evaluate('XGBoost', df_with_extra, df_with_extra['pIC50'])
model_lgbm, rmse_lgbm, r2_lgbm = train_and_evaluate('LightGBM', df_with_extra, df_with_extra['pIC50'])


== RandomForest 결과 ==
RMSE: 0.606
R^2: 0.800

== XGBoost 결과 ==
RMSE: 0.599
R^2: 0.805

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.020016 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2205
[LightGBM] [Info] Number of data points in the train set: 3739, number of used features: 971
[LightGBM] [Info] Start training from score 6.827843
== LightGBM 결과 ==
RMSE: 0.611
R^2: 0.797



In [ ]:
import os
import numpy as np
import pandas as pd
from rdkit import Chem
from rdkit.Chem import Descriptors, AllChem

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

# 파일 경로 및 타입 컬럼 정의
file_infos = {
    "ChEMBL": {
        "path": " /summer_conference/open/ChEMBL_ASK1(IC50)_clean.csv",
        "type_col": "Standard Type"
    },
    "CAS": {
        "path": " /summer_conference/open/CAS_clean.csv",
        "type_col": "Assay Parameter"
    },
    "PubChem": {
        "path": " /summer_conference/open/Pubchem_clean.csv",
        "type_col": "Activity_Type"
    }
}

# IC50 데이터 병합
dfs = []
for source, info in file_infos.items():
    df = pd.read_csv(info["path"], encoding='cp1252')
    type_col = info["type_col"]

    # pIC50 컬럼 찾기
    if 'pIC50' in df.columns:
        pic50_col = 'pIC50'
    elif 'pX Value' in df.columns:
        pic50_col = 'pX Value'
    elif 'Assay Parameter' in df.columns:
        pic50_col = 'Assay Parameter'
    else:
        raise ValueError(f"{source}: pIC50 컬럼이 없습니다.")

    df = df[df[type_col].str.upper() == 'IC50']
    df = df[['SMILES', pic50_col]].copy()
    df.rename(columns={pic50_col: 'pIC50'}, inplace=True)
    df['Source'] = source
    dfs.append(df)

combined_df = pd.concat(dfs, ignore_index=True)

# SMILES → mol 변환
def safe_mol(smiles):
    try:
        return Chem.MolFromSmiles(smiles)
    except:
        return None

combined_df['mol'] = combined_df['SMILES'].apply(safe_mol)
combined_df = combined_df.dropna(subset=['mol'])

# Num_N, TPSA 피처 생성
def extract_features(mol):
    return {
        'Num_N': sum(atom.GetSymbol() == 'N' for atom in mol.GetAtoms()),
        'TPSA': Descriptors.TPSA(mol)
    }

extra_feats = pd.DataFrame([extract_features(mol) for mol in combined_df['mol']])
df_with_extra = pd.concat([combined_df.reset_index(drop=True), extra_feats], axis=1)

# Morgan Fingerprint 계산
def mol_to_morgan_fp(mol, radius=2, n_bits=2048):
    if mol is None:
        return np.zeros(n_bits, dtype=int)
    fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius, nBits=n_bits)
    arr = np.zeros((n_bits,), dtype=int)
    AllChem.DataStructs.ConvertToNumpyArray(fp, arr)
    return arr

fp_array = np.array([mol_to_morgan_fp(mol) for mol in df_with_extra['mol']])
fp_df = pd.DataFrame(fp_array, columns=[f'FP_{i}' for i in range(fp_array.shape[1])])
df_with_extra = pd.concat([df_with_extra, fp_df], axis=1)

# 커스텀 평가 함수 (Normalized RMSE + R2 → Score)
def evaluate_custom_score(y_true, y_pred):
    ic50_true = 10 ** (-y_true)
    ic50_pred = 10 ** (-y_pred)

    rmse_ic50 = np.sqrt(mean_squared_error(ic50_true, ic50_pred))
    nrmse = rmse_ic50 / np.mean(ic50_true)
    A = nrmse
    B = r2_score(y_true, y_pred)

    score = 0.4 * (1 - min(A, 1)) + 0.6 * B

    print(f"🔍 Normalized RMSE (A): {A:.3f}")
    print(f"🔍 R² (B): {B:.3f}")
    print(f"✅ 최종 Score: {score:.3f}\n")
    return A, B, score

# 학습 및 평가 함수
def train_and_evaluate(model_name, X, y):
    X = X.drop(columns=['SMILES', 'pIC50', 'mol', 'Source'], errors='ignore')
    valid_idx = y.notnull() & X.notnull().all(axis=1)
    X_train, X_test, y_train, y_test = train_test_split(
        X[valid_idx], y[valid_idx], test_size=0.2, random_state=42)

    if model_name == 'RandomForest':
        model = RandomForestRegressor(n_estimators=100, random_state=42)
    elif model_name == 'XGBoost':
        model = XGBRegressor(n_estimators=100, random_state=42, verbosity=0)
    elif model_name == 'LightGBM':
        model = LGBMRegressor(n_estimators=100, random_state=42)
    else:
        raise ValueError(f"모델 '{model_name}'은 지원되지 않습니다.")

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)

    print(f"📊 [{model_name}] 평가 결과")
    print(f"RMSE: {rmse:.3f}")
    print(f"R²: {r2:.3f}")

    evaluate_custom_score(y_test, y_pred)
    return model

# 모델별 학습 및 평가
model_rf = train_and_evaluate('RandomForest', df_with_extra, df_with_extra['pIC50'])
model_xgb = train_and_evaluate('XGBoost', df_with_extra, df_with_extra['pIC50'])
model_lgbm = train_and_evaluate('LightGBM', df_with_extra, df_with_extra['pIC50'])


[10:32:19] DEPRECATION WARNING: please use MorganGenerator
[10:32:19] DEPRECATION WARNING: please use MorganGenerator
[10:32:19] DEPRECATION WARNING: please use MorganGenerator
[10:32:19] DEPRECATION WARNING: please use MorganGenerator
[10:32:19] DEPRECATION WARNING: please use MorganGenerator
[10:32:19] DEPRECATION WARNING: please use MorganGenerator
[10:32:19] DEPRECATION WARNING: please use MorganGenerator
[10:32:19] DEPRECATION WARNING: please use MorganGenerator
[10:32:19] DEPRECATION WARNING: please use MorganGenerator
[10:32:19] DEPRECATION WARNING: please use MorganGenerator
[10:32:19] DEPRECATION WARNING: please use MorganGenerator
[10:32:19] DEPRECATION WARNING: please use MorganGenerator
[10:32:19] DEPRECATION WARNING: please use MorganGenerator
[10:32:19] DEPRECATION WARNING: please use MorganGenerator
[10:32:19] DEPRECATION WARNING: please use MorganGenerator
[10:32:19] DEPRECATION WARNING: please use MorganGenerator
[10:32:19] DEPRECATION WARNING: please use MorganGenerat

📊 [RandomForest] 평가 결과
RMSE: 0.849
R²: 0.471
🔍 Normalized RMSE (A): 5.884
🔍 R² (B): 0.471
✅ 최종 Score: 0.282

📊 [XGBoost] 평가 결과
RMSE: 0.899
R²: 0.407
🔍 Normalized RMSE (A): 5.815
🔍 R² (B): 0.407
✅ 최종 Score: 0.244

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001484 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 627
[LightGBM] [Info] Number of data points in the train set: 649, number of used features: 265
[LightGBM] [Info] Start training from score 6.667127
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGB

In [12]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from rdkit import Chem
from rdkit.Chem import Descriptors, AllChem, rdMolDescriptors
import pandas as pd
import numpy as np
import os

# 파일 정보
file_infos = {
    "ChEMBL": {
        "path": " /summer_conference/open/ChEMBL_ASK1(IC50)_clean.csv",
        "type_col": "Standard Type"
    },
    "CAS": {
        "path": " /summer_conference/open/CAS_clean.csv",
        "type_col": "Assay Parameter"
    },
    "PubChem": {
        "path": " /summer_conference/open/Pubchem_clean.csv",
        "type_col": "Activity_Type"
    }
}

# 데이터 불러오기 및 정리
dfs = []
for source, info in file_infos.items():
    df = pd.read_csv(info["path"], encoding='cp1252')
    type_col = info["type_col"]

    if 'pIC50' in df.columns:
        pic50_col = 'pIC50'
    elif 'pX Value' in df.columns:
        pic50_col = 'pX Value'
    elif 'Assay Parameter' in df.columns:
        pic50_col = 'Assay Parameter'
    else:
        raise ValueError(f"{source}: pIC50 컬럼이 없습니다.")

    df = df[df[type_col].str.upper() == 'IC50']
    df = df[['SMILES', pic50_col]].copy()
    df.rename(columns={pic50_col: 'pIC50'}, inplace=True)
    df['Source'] = source
    dfs.append(df)

combined_df = pd.concat(dfs, ignore_index=True)

# Mol 생성
def safe_mol(smiles):
    try:
        return Chem.MolFromSmiles(smiles)
    except:
        return None

combined_df['mol'] = combined_df['SMILES'].apply(safe_mol)
combined_df = combined_df.dropna(subset=['mol'])

# 간단 피처
def extract_features(mol):
    return {
        'Num_N': sum(atom.GetSymbol() == 'N' for atom in mol.GetAtoms()),
    }

extra_feats = pd.DataFrame([extract_features(mol) for mol in combined_df['mol']])
df_with_extra = pd.concat([combined_df.reset_index(drop=True), extra_feats], axis=1)

# Morgan Fingerprint
def mol_to_morgan_fp(mol, radius=2, n_bits=2048):
    if mol is None:
        return np.zeros(n_bits, dtype=int)
    fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius, nBits=n_bits)
    arr = np.zeros((n_bits,), dtype=int)
    AllChem.DataStructs.ConvertToNumpyArray(fp, arr)
    return arr

fp_array = np.array([mol_to_morgan_fp(mol) for mol in df_with_extra['mol']])
fp_df = pd.DataFrame(fp_array, columns=[f'FP_{i}' for i in range(fp_array.shape[1])])

# 분자 특성 피처
def featurize_full(mol):
    feats = {
        'MolWt': Descriptors.MolWt(mol),
        'MolLogP': Descriptors.MolLogP(mol),
        'TPSA': Descriptors.TPSA(mol),
        'NumHDonors': Descriptors.NumHDonors(mol),
        'NumHAcceptors': Descriptors.NumHAcceptors(mol),
        'NumRotatableBonds': Descriptors.NumRotatableBonds(mol),
        'NumValenceElectrons': Descriptors.NumValenceElectrons(mol),
        'HeavyAtomCount': Descriptors.HeavyAtomCount(mol),
        'NumAliphaticRings': Descriptors.NumAliphaticRings(mol),
        'NumAromaticRings': rdMolDescriptors.CalcNumAromaticRings(mol),
        'NumRings': rdMolDescriptors.CalcNumRings(mol),
        'NumCharges': sum(a.GetFormalCharge() != 0 for a in mol.GetAtoms()),
        'NumCarbonAtoms': sum(a.GetSymbol() == 'C' for a in mol.GetAtoms())
    }

    atom_types = ['C', 'H', 'N', 'O', 'Cl', 'S', 'F', 'Br', 'I', 'B', 'Si']
    for atom in atom_types:
        feats[f'Num_{atom}'] = sum(a.GetSymbol() == atom for a in mol.GetAtoms())

    fg_smarts = {
        'Amine_primary': '[NX3;H2]',
        'Amine_secondary': '[NX3;H1]',
        'Amine_tertiary': '[NX3;H0]',
        'Alcohol': '[OX2H]',
        'Aldehyde': '[CX3H1](=O)[#6]',
        'Ketone': '[#6][CX3](=O)[#6]',
        'Carboxylic_acid': '[CX3](=O)[OX2H1]',
        'Ester': '[CX3](=O)[OX2][#6]',
        'Nitrile': '[CX2]#N'
    }
    for name, patt in fg_smarts.items():
        smarts = Chem.MolFromSmarts(patt)
        feats[f'FG_{name}'] = len(mol.GetSubstructMatches(smarts))

    return feats

full_feats = [featurize_full(mol) for mol in df_with_extra['mol']]
full_feat_df = pd.DataFrame(full_feats)

# 병합 (TPSA 중복 제거)
X_all = pd.concat([
    df_with_extra[['SMILES', 'pIC50', 'mol', 'Source', 'Num_N']],
    full_feat_df.reset_index(drop=True),
    fp_df.reset_index(drop=True)
], axis=1)

# 평가 함수
def evaluate_custom_score(y_true, y_pred):
    ic50_true = 10 ** (-y_true)
    ic50_pred = 10 ** (-y_pred)
    rmse_ic50 = np.sqrt(mean_squared_error(ic50_true, ic50_pred))
    nrmse = rmse_ic50 / np.mean(ic50_true)
    A = nrmse
    B = r2_score(y_true, y_pred)
    score = 0.4 * (1 - min(A, 1)) + 0.6 * B
    print(f"🔍 Normalized RMSE (A): {A:.3f}")
    print(f"🔍 R² (B): {B:.3f}")
    print(f"✅ 최종 Score: {score:.3f}\n")
    return A, B, score

# 학습 및 평가
def train_and_evaluate(model_name, X, y):
    X = X.drop(columns=['SMILES', 'pIC50', 'mol', 'Source'], errors='ignore')
    valid_idx = y.notnull() & X.notnull().all(axis=1)
    X_train, X_test, y_train, y_test = train_test_split(
        X[valid_idx], y[valid_idx], test_size=0.2, random_state=42)

    if model_name == 'RandomForest':
        model = RandomForestRegressor(random_state=42)
    elif model_name == 'XGBoost':
        model = XGBRegressor(random_state=42, verbosity=0)
    elif model_name == 'LightGBM':
        model = LGBMRegressor(random_state=42)
    else:
        raise ValueError(f"모델 '{model_name}'은 지원되지 않습니다.")

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    print(f"\n📊 [{model_name}] 평가 결과")
    print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_pred)):.3f}")
    print(f"R²: {r2_score(y_test, y_pred):.3f}")
    evaluate_custom_score(y_test, y_pred)
    return model
X_all = X_all.loc[:, ~X_all.columns.duplicated()]
# 디폴트 모델 학습 실행
rf_model = train_and_evaluate('RandomForest', X_all, X_all['pIC50'])
xgb_model = train_and_evaluate('XGBoost', X_all, X_all['pIC50'])
lgbm_model = train_and_evaluate('LightGBM', X_all, X_all['pIC50'])


[18:12:09] DEPRECATION WARNING: please use MorganGenerator
[18:12:09] DEPRECATION WARNING: please use MorganGenerator
[18:12:09] DEPRECATION WARNING: please use MorganGenerator
[18:12:09] DEPRECATION WARNING: please use MorganGenerator
[18:12:09] DEPRECATION WARNING: please use MorganGenerator
[18:12:09] DEPRECATION WARNING: please use MorganGenerator
[18:12:09] DEPRECATION WARNING: please use MorganGenerator
[18:12:09] DEPRECATION WARNING: please use MorganGenerator
[18:12:09] DEPRECATION WARNING: please use MorganGenerator
[18:12:09] DEPRECATION WARNING: please use MorganGenerator
[18:12:09] DEPRECATION WARNING: please use MorganGenerator
[18:12:09] DEPRECATION WARNING: please use MorganGenerator
[18:12:09] DEPRECATION WARNING: please use MorganGenerator
[18:12:09] DEPRECATION WARNING: please use MorganGenerator
[18:12:09] DEPRECATION WARNING: please use MorganGenerator
[18:12:09] DEPRECATION WARNING: please use MorganGenerator
[18:12:09] DEPRECATION WARNING: please use MorganGenerat


📊 [RandomForest] 평가 결과
RMSE: 0.613
R²: 0.796
🔍 Normalized RMSE (A): 3.842
🔍 R² (B): 0.796
✅ 최종 Score: 0.478


📊 [XGBoost] 평가 결과
RMSE: 0.603
R²: 0.802
🔍 Normalized RMSE (A): 0.903
🔍 R² (B): 0.802
✅ 최종 Score: 0.520

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.019539 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3014
[LightGBM] [Info] Number of data points in the train set: 3739, number of used features: 996
[LightGBM] [Info] Start training from score 6.827843

📊 [LightGBM] 평가 결과
RMSE: 0.612
R²: 0.796
🔍 Normalized RMSE (A): 3.980
🔍 R² (B): 0.796
✅ 최종 Score: 0.478



In [2]:
from sklearn.model_selection import GridSearchCV

def train_with_grid_search(model_name, X, y):
    X = X.drop(columns=['SMILES', 'pIC50', 'mol', 'Source'], errors='ignore')
    valid_idx = y.notnull() & X.notnull().all(axis=1)
    X_train, X_test, y_train, y_test = train_test_split(
        X[valid_idx], y[valid_idx], test_size=0.2, random_state=42)

    if model_name == 'RandomForest':
        model = RandomForestRegressor(random_state=42)
        param_grid = {
            'n_estimators': [100, 200],
            'max_depth': [None, 10, 20],
            'max_features': ['auto', 'sqrt'],
            'min_samples_split': [2, 5]
        }
    elif model_name == 'XGBoost':
        model = XGBRegressor(random_state=42, verbosity=0)
        param_grid = {
            'n_estimators': [100, 200],
            'max_depth': [3, 6],
            'learning_rate': [0.01, 0.1],
            'subsample': [0.8, 1]
        }
    elif model_name == 'LightGBM':
        model = LGBMRegressor(random_state=42)
        param_grid = {
            'n_estimators': [100, 200],
            'max_depth': [-1, 10],
            'learning_rate': [0.01, 0.1],
            'num_leaves': [31, 50]
        }
    else:
        raise ValueError(f"모델 '{model_name}'은 지원되지 않습니다.")

    grid_search = GridSearchCV(model, param_grid, cv=3, scoring='neg_mean_squared_error', n_jobs=-1)
    grid_search.fit(X_train, y_train)

    best_model = grid_search.best_estimator_
    y_pred = best_model.predict(X_test)

    print(f"\n📊 [{model_name} - GridSearchCV] 최적 하이퍼파라미터: {grid_search.best_params_}")
    print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_pred)):.3f}")
    print(f"R²: {r2_score(y_test, y_pred):.3f}")
    evaluate_custom_score(y_test, y_pred)

    return best_model


Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.


In [6]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint, uniform

def train_with_random_search(model_name, X, y, n_iter=20):
    X = X.drop(columns=['SMILES', 'pIC50', 'mol', 'Source'], errors='ignore')
    valid_idx = y.notnull() & X.notnull().all(axis=1)
    X_train, X_test, y_train, y_test = train_test_split(
        X[valid_idx], y[valid_idx], test_size=0.2, random_state=42)

    if model_name == 'RandomForest':
        model = RandomForestRegressor(random_state=42)
        param_dist = {
            'n_estimators': randint(50, 300),
            'max_depth': randint(5, 30),
            'max_features': ['auto', 'sqrt', 'log2'],
            'min_samples_split': randint(2, 10)
        }
    elif model_name == 'XGBoost':
        model = XGBRegressor(random_state=42, verbosity=0)
        param_dist = {
            'n_estimators': randint(50, 300),
            'max_depth': randint(3, 10),
            'learning_rate': uniform(0.01, 0.3),
            'subsample': uniform(0.7, 0.3)
        }
    elif model_name == 'LightGBM':
        model = LGBMRegressor(random_state=42)
        param_dist = {
            'n_estimators': randint(50, 300),
            'max_depth': randint(-1, 30),  # -1은 무제한
            'learning_rate': uniform(0.01, 0.3),
            'num_leaves': randint(20, 60)
        }
    else:
        raise ValueError(f"모델 '{model_name}'은 지원되지 않습니다.")

    random_search = RandomizedSearchCV(model, param_distributions=param_dist, 
                                       n_iter=n_iter, cv=3, scoring='neg_mean_squared_error',
                                       n_jobs=-1, random_state=42)
    random_search.fit(X_train, y_train)

    best_model = random_search.best_estimator_
    y_pred = best_model.predict(X_test)

    print(f"\n📊 [{model_name} - RandomizedSearchCV] 최적 하이퍼파라미터: {random_search.best_params_}")
    print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_pred)):.3f}")
    print(f"R²: {r2_score(y_test, y_pred):.3f}")
    evaluate_custom_score(y_test, y_pred)

    return best_model


In [7]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from rdkit import Chem
from rdkit.Chem import Descriptors, AllChem, rdMolDescriptors
import pandas as pd
import numpy as np
import os

# 파일 정보
file_infos = {
    "ChEMBL": {
        "path": " /summer_conference/open/ChEMBL_ASK1(IC50)_clean.csv",
        "type_col": "Standard Type"
    },
    "CAS": {
        "path": " /summer_conference/open/CAS_clean.csv",
        "type_col": "Assay Parameter"
    },
    "PubChem": {
        "path": " /summer_conference/open/Pubchem_clean.csv",
        "type_col": "Activity_Type"
    }
}

# 데이터 불러오기 및 정리
dfs = []
for source, info in file_infos.items():
    df = pd.read_csv(info["path"], encoding='cp1252')
    type_col = info["type_col"]

    if 'pIC50' in df.columns:
        pic50_col = 'pIC50'
    elif 'pX Value' in df.columns:
        pic50_col = 'pX Value'
    elif 'Assay Parameter' in df.columns:
        pic50_col = 'Assay Parameter'
    else:
        raise ValueError(f"{source}: pIC50 컬럼이 없습니다.")

    df = df[df[type_col].str.upper() == 'IC50']
    df = df[['SMILES', pic50_col]].copy()
    df.rename(columns={pic50_col: 'pIC50'}, inplace=True)
    df['Source'] = source
    dfs.append(df)

combined_df = pd.concat(dfs, ignore_index=True)

# Mol 생성
def safe_mol(smiles):
    try:
        return Chem.MolFromSmiles(smiles)
    except:
        return None

combined_df['mol'] = combined_df['SMILES'].apply(safe_mol)
combined_df = combined_df.dropna(subset=['mol'])

# 간단 피처
def extract_features(mol):
    return {
        'Num_N': sum(atom.GetSymbol() == 'N' for atom in mol.GetAtoms()),
    }

extra_feats = pd.DataFrame([extract_features(mol) for mol in combined_df['mol']])
df_with_extra = pd.concat([combined_df.reset_index(drop=True), extra_feats], axis=1)

# Morgan Fingerprint
def mol_to_morgan_fp(mol, radius=2, n_bits=2048):
    if mol is None:
        return np.zeros(n_bits, dtype=int)
    fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius, nBits=n_bits)
    arr = np.zeros((n_bits,), dtype=int)
    AllChem.DataStructs.ConvertToNumpyArray(fp, arr)
    return arr

fp_array = np.array([mol_to_morgan_fp(mol) for mol in df_with_extra['mol']])
fp_df = pd.DataFrame(fp_array, columns=[f'FP_{i}' for i in range(fp_array.shape[1])])

# 분자 특성 피처
def featurize_full(mol):
    feats = {
        'MolWt': Descriptors.MolWt(mol),
        'MolLogP': Descriptors.MolLogP(mol),
        'TPSA': Descriptors.TPSA(mol),
        'NumHDonors': Descriptors.NumHDonors(mol),
        'NumHAcceptors': Descriptors.NumHAcceptors(mol),
        'NumRotatableBonds': Descriptors.NumRotatableBonds(mol),
        'NumValenceElectrons': Descriptors.NumValenceElectrons(mol),
        'HeavyAtomCount': Descriptors.HeavyAtomCount(mol),
        'NumAliphaticRings': Descriptors.NumAliphaticRings(mol),
        'NumAromaticRings': rdMolDescriptors.CalcNumAromaticRings(mol),
        'NumRings': rdMolDescriptors.CalcNumRings(mol),
        'NumCharges': sum(a.GetFormalCharge() != 0 for a in mol.GetAtoms()),
        'NumCarbonAtoms': sum(a.GetSymbol() == 'C' for a in mol.GetAtoms())
    }

    atom_types = ['C', 'H', 'N', 'O', 'Cl', 'S', 'F', 'Br', 'I', 'B', 'Si']
    for atom in atom_types:
        feats[f'Num_{atom}'] = sum(a.GetSymbol() == atom for a in mol.GetAtoms())

    fg_smarts = {
        'Amine_primary': '[NX3;H2]',
        'Amine_secondary': '[NX3;H1]',
        'Amine_tertiary': '[NX3;H0]',
        'Alcohol': '[OX2H]',
        'Aldehyde': '[CX3H1](=O)[#6]',
        'Ketone': '[#6][CX3](=O)[#6]',
        'Carboxylic_acid': '[CX3](=O)[OX2H1]',
        'Ester': '[CX3](=O)[OX2][#6]',
        'Nitrile': '[CX2]#N'
    }
    for name, patt in fg_smarts.items():
        smarts = Chem.MolFromSmarts(patt)
        feats[f'FG_{name}'] = len(mol.GetSubstructMatches(smarts))

    return feats

full_feats = [featurize_full(mol) for mol in df_with_extra['mol']]
full_feat_df = pd.DataFrame(full_feats)

# 병합 (TPSA 중복 제거)
X_all = pd.concat([
    df_with_extra[['SMILES', 'pIC50', 'mol', 'Source', 'Num_N']],
    full_feat_df.reset_index(drop=True),
    fp_df.reset_index(drop=True)
], axis=1)

# 평가 함수
def evaluate_custom_score(y_true, y_pred):
    ic50_true = 10 ** (-y_true)
    ic50_pred = 10 ** (-y_pred)
    rmse_ic50 = np.sqrt(mean_squared_error(ic50_true, ic50_pred))
    nrmse = rmse_ic50 / np.mean(ic50_true)
    A = nrmse
    B = r2_score(y_true, y_pred)
    score = 0.4 * (1 - min(A, 1)) + 0.6 * B
    print(f"🔍 Normalized RMSE (A): {A:.3f}")
    print(f"🔍 R² (B): {B:.3f}")
    print(f"✅ 최종 Score")
    print(score)

[16:07:20] DEPRECATION WARNING: please use MorganGenerator
[16:07:20] DEPRECATION WARNING: please use MorganGenerator
[16:07:20] DEPRECATION WARNING: please use MorganGenerator
[16:07:20] DEPRECATION WARNING: please use MorganGenerator
[16:07:20] DEPRECATION WARNING: please use MorganGenerator
[16:07:20] DEPRECATION WARNING: please use MorganGenerator
[16:07:20] DEPRECATION WARNING: please use MorganGenerator
[16:07:20] DEPRECATION WARNING: please use MorganGenerator
[16:07:20] DEPRECATION WARNING: please use MorganGenerator
[16:07:20] DEPRECATION WARNING: please use MorganGenerator
[16:07:20] DEPRECATION WARNING: please use MorganGenerator
[16:07:20] DEPRECATION WARNING: please use MorganGenerator
[16:07:20] DEPRECATION WARNING: please use MorganGenerator
[16:07:20] DEPRECATION WARNING: please use MorganGenerator
[16:07:20] DEPRECATION WARNING: please use MorganGenerator
[16:07:20] DEPRECATION WARNING: please use MorganGenerator
[16:07:20] DEPRECATION WARNING: please use MorganGenerat

In [8]:
# Grid Search 예시
best_rf_grid = train_with_grid_search('RandomForest', X_all, X_all['pIC50'])

# Random Search 예시
best_rf_random = train_with_random_search('RandomForest', X_all, X_all['pIC50'], n_iter=30)


Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.


/Users/oseli/opt/anaconda3/envs/rdkit-env/lib/python3.9/site-packages/sklearn/ensemble/_forest.py:416: FutureWarning: `max_features='auto'` has been deprecated in 1.1 and will be removed in 1.3. To keep the past behaviour, explicitly set `max_features=1.0` or remove this parameter as it is also the default value for RandomForestRegressors and ExtraTreesRegressors.
  warn(
/Users/oseli/opt/anaconda3/envs/rdkit-env/lib/python3.9/site-packages/sklearn/ensemble/_forest.py:416: FutureWarning: `max_features='auto'` has been deprecated in 1.1 and will be removed in 1.3. To keep the past behaviour, explicitly set `max_features=1.0` or remove this parameter as it is also the default value for RandomForestRegressors and ExtraTreesRegressors.
  warn(
/Users/oseli/opt/anaconda3/envs/rdkit-env/lib/python3.9/site-packages/sklearn/ensemble/_forest.py:416: FutureWarning: `max_features='auto'` has been deprecated in 1.1 and will be removed in 1.3. To keep the past behaviour, explicitly set `max_feature


📊 [RandomForest - GridSearchCV] 최적 하이퍼파라미터: {'max_depth': None, 'max_features': 'sqrt', 'min_samples_split': 2, 'n_estimators': 200}
RMSE: 0.601
R²: 0.804
🔍 Normalized RMSE (A): 3.195
🔍 R² (B): 0.804
✅ 최종 Score


/Users/oseli/opt/anaconda3/envs/rdkit-env/lib/python3.9/site-packages/sklearn/ensemble/_forest.py:416: FutureWarning: `max_features='auto'` has been deprecated in 1.1 and will be removed in 1.3. To keep the past behaviour, explicitly set `max_features=1.0` or remove this parameter as it is also the default value for RandomForestRegressors and ExtraTreesRegressors.
  warn(
/Users/oseli/opt/anaconda3/envs/rdkit-env/lib/python3.9/site-packages/sklearn/ensemble/_forest.py:416: FutureWarning: `max_features='auto'` has been deprecated in 1.1 and will be removed in 1.3. To keep the past behaviour, explicitly set `max_features=1.0` or remove this parameter as it is also the default value for RandomForestRegressors and ExtraTreesRegressors.
  warn(
/Users/oseli/opt/anaconda3/envs/rdkit-env/lib/python3.9/site-packages/sklearn/ensemble/_forest.py:416: FutureWarning: `max_features='auto'` has been deprecated in 1.1 and will be removed in 1.3. To keep the past behaviour, explicitly set `max_feature


📊 [RandomForest - RandomizedSearchCV] 최적 하이퍼파라미터: {'max_depth': 25, 'max_features': 'sqrt', 'min_samples_split': 2, 'n_estimators': 237}
RMSE: 0.598
R²: 0.806
🔍 Normalized RMSE (A): 3.256
🔍 R² (B): 0.806
✅ 최종 Score


In [ ]:
# Grid Search 예시
# 중복 컬럼명 제거 (맨 처음 등장하는 컬럼만 남기고 제거)
X_all = X_all.loc[:, ~X_all.columns.duplicated()]

best_xgb_grid = train_with_grid_search('XGBoost', X_all, X_all['pIC50'])

# Random Search 예시
best_xgb_random = train_with_random_search('XGBoost', X_all, X_all['pIC50'], n_iter=30)


Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.


In [9]:
def train_and_tune_model(model_name, X, y):
    X = X.drop(columns=['SMILES', 'pIC50', 'mol', 'Source'], errors='ignore')
    valid_idx = y.notnull() & X.notnull().all(axis=1)
    X_train, X_test, y_train, y_test = train_test_split(
        X[valid_idx], y[valid_idx], test_size=0.2, random_state=42)

    if model_name == 'RandomForest':
        model = RandomForestRegressor(random_state=42)
    elif model_name == 'XGBoost':
        model = XGBRegressor(random_state=42, verbosity=0)
    elif model_name == 'LightGBM':
        model = LGBMRegressor(random_state=42)
    else:
        raise ValueError(f"모델 '{model_name}'은 지원되지 않습니다.")

    print(f"\n🛠 [{model_name}] 기본 하이퍼파라미터로 모델 학습 시작...")
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    print(f"\n📊 [{model_name}] 기본 모델 평가 결과")
    print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_pred)):.3f}")
    print(f"R²: {r2_score(y_test, y_pred):.3f}")
    evaluate_custom_score(y_test, y_pred)

    return model


In [ ]:
import xgboost
print(xgboost.__file__)


/Users/oseli/opt/anaconda3/envs/rdkit-env/lib/python3.9/site-packages/xgboost/__init__.py


In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import numpy as np

# X, y 준비 완료된 상태라고 가정
# (예: df_with_extra에서 feature, target 분리 후 결측치 처리 완료)

# 데이터 스케일링 (딥러닝에 필수)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# train/test 분할
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

# 모델 정의
model = Sequential([
    Dense(128, activation='relu', input_shape=(X_train.shape[1],)),
    Dropout(0.2),
    Dense(64, activation='relu'),
    Dropout(0.2),
    Dense(1)  # 회귀라 출력 1개
])

# 컴파일
model.compile(optimizer=Adam(learning_rate=0.001),
              loss='mse',
              metrics=['mae'])

# 학습
history = model.fit(
    X_train, y_train,
    validation_split=0.2,
    epochs=50,
    batch_size=32,
    verbose=2
)

# 평가
test_loss, test_mae = model.evaluate(X_test, y_test, verbose=2)
print(f"Test MSE: {test_loss:.4f}, Test MAE: {test_mae:.4f}")

# 예측
y_pred = model.predict(X_test).flatten()



ModuleNotFoundError: No module named 'tensorflow'

In [3]:
#여기서 시작------------------------------------------------------
import os
import random

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from rdkit import Chem
from rdkit.Chem import Descriptors, rdMolDescriptors, AllChem, rdFMCS
from rdkit.DataStructs import TanimotoSimilarity

from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
from scipy.spatial.distance import squareform
from sklearn.decomposition import PCA

def calc_features(smiles, pIC50=7.0):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    try:
        MolWt = Descriptors.MolWt(mol)
        LogP = Descriptors.MolLogP(mol)
        TPSA = Descriptors.TPSA(mol)
        HeavyAtoms = Descriptors.HeavyAtomCount(mol)
        BEI = pIC50 / (MolWt + 1e-6)
        LLE = pIC50 - LogP
        LE = pIC50 / (HeavyAtoms + 1e-6)
        TPSA_per_MolWt = TPSA / (MolWt + 1e-6)
        return [LE, BEI, LLE, TPSA_per_MolWt]
    except:
        return None

def pIC50_to_IC50(p): return 10 ** (-p)

pub = pd.read_csv(" /summer_conference/open/Pubchem_ASK1.csv", low_memory=False)
pub = pub[pub['Activity_Value'] > 0].copy()
pub['pIC50'] = -np.log10(pub['Activity_Value'])
pub = pub[['SMILES', 'pIC50']].dropna()

chem = pd.read_csv(" /summer_conference/open/ChEMBL_ASK1(IC50).csv", sep=';')
chem = chem[['Smiles', 'pChEMBL Value']].dropna()
chem = chem.rename(columns={'Smiles': 'SMILES', 'pChEMBL Value': 'pIC50'})

df = pd.concat([pub, chem], ignore_index=True).drop_duplicates('SMILES').reset_index(drop=True)

In [7]:
features_list = []
for smi in df['SMILES']:
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        continue
    features = {}
    fp = rdMolDescriptors.GetMorganFingerprintAsBitVect(mol, radius=2, nBits=1024)
    features.update({f'FP_{i}': int(bit) for i, bit in enumerate(fp.ToBitString())})
    features_list.append(features)


[20:21:25] DEPRECATION WARNING: please use MorganGenerator
[20:21:25] DEPRECATION WARNING: please use MorganGenerator
[20:21:25] DEPRECATION WARNING: please use MorganGenerator
[20:21:25] DEPRECATION WARNING: please use MorganGenerator
[20:21:25] DEPRECATION WARNING: please use MorganGenerator
[20:21:25] DEPRECATION WARNING: please use MorganGenerator
[20:21:25] DEPRECATION WARNING: please use MorganGenerator
[20:21:25] DEPRECATION WARNING: please use MorganGenerator
[20:21:25] DEPRECATION WARNING: please use MorganGenerator
[20:21:25] DEPRECATION WARNING: please use MorganGenerator
[20:21:25] DEPRECATION WARNING: please use MorganGenerator
[20:21:25] DEPRECATION WARNING: please use MorganGenerator
[20:21:25] DEPRECATION WARNING: please use MorganGenerator
[20:21:25] DEPRECATION WARNING: please use MorganGenerator
[20:21:25] DEPRECATION WARNING: please use MorganGenerator
[20:21:25] DEPRECATION WARNING: please use MorganGenerator
[20:21:25] DEPRECATION WARNING: please use MorganGenerat

In [10]:
# [1] 라이브러리
import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem import Descriptors
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor

# [2] RDKit 파생 피처 계산 함수
def calc_features(smiles, pIC50=7.0):  # pIC50은 계산용 placeholder
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    try:
        MolWt = Descriptors.MolWt(mol)
        LogP = Descriptors.MolLogP(mol)
        TPSA = Descriptors.TPSA(mol)
        HeavyAtoms = Descriptors.HeavyAtomCount(mol)
        BEI = pIC50 / (MolWt + 1e-6)
        LLE = pIC50 - LogP
        LE = pIC50 / (HeavyAtoms + 1e-6)
        TPSA_per_MolWt = TPSA / (MolWt + 1e-6)
        return [LE, BEI, LLE, TPSA_per_MolWt]
    except:
        return None

# [3] pIC50 → IC50 변환 함수
def pIC50_to_IC50(pIC50_array):
    return 10 ** (-pIC50_array)

# [4] 학습 데이터 로딩 및 병합
pub = pd.read_csv(" /summer_conference/open/Pubchem_ASK1.csv", low_memory=False)
pub = pub[pub['Activity_Value'] > 0].copy()
pub['pIC50'] = -np.log10(pub['Activity_Value'])
pub = pub[['SMILES', 'pIC50']].dropna()

chem = pd.read_csv(" /summer_conference/open/ChEMBL_ASK1(IC50).csv", sep=';')
chem = chem[['Smiles', 'pChEMBL Value']].dropna()
chem = chem.rename(columns={'Smiles': 'SMILES', 'pChEMBL Value': 'pIC50'})

df = pd.concat([pub, chem], ignore_index=True).drop_duplicates('SMILES').reset_index(drop=True)

# [5] 파생 피처 생성
train_features = df.apply(lambda row: calc_features(row['SMILES'], row['pIC50']), axis=1)
train_df = pd.DataFrame(train_features.tolist(), columns=['LE', 'BEI', 'LLE', 'TPSA_per_MolWt'])
train_df['pIC50'] = df['pIC50']

# [6] 학습/검증 분할
X = train_df[['LE', 'BEI', 'LLE', 'TPSA_per_MolWt']].values
y = train_df['pIC50'].values
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# [7] 모델 학습
model = RandomForestRegressor(n_estimators=300, max_depth=12, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)

# [8] 테스트 데이터 불러오기
test_df = pd.read_csv(" /summer_conference/open/test.csv")
test_features = test_df['Smiles'].apply(lambda s: calc_features(s))
test_feature_df = pd.DataFrame(test_features.tolist(), columns=['LE', 'BEI', 'LLE', 'TPSA_per_MolWt'])

# [9] 유효한 테스트 샘플 필터링 및 예측
valid_test_mask = test_feature_df.notnull().all(axis=1)
X_test_final = test_feature_df.loc[valid_test_mask].values
test_preds_pIC50 = model.predict(X_test_final)
test_preds_IC50 = pIC50_to_IC50(test_preds_pIC50)

# [10] sample_submission 불러와서 예측값 넣기
sample_sub = pd.read_csv(" /summer_conference/open/sample_submission.csv")
pred_df = pd.DataFrame({
    'ID': test_df.loc[valid_test_mask, 'ID'].values,
    'ASK1_IC50_nM': test_preds_IC50
})
submission = sample_sub[['ID']].merge(pred_df, on='ID', how='left')

# [11] 결측값 처리 (평균 IC50으로)
submission['ASK1_IC50_nM'].fillna(pIC50_to_IC50(y.mean()), inplace=True)

# [12] 제출 파일 저장
submission.to_csv("rf_submission.csv", index=False)
print("✅ 제출용 파일 생성 완료 → rf_submission.csv")

# [7-1] 내부 검증 성능 평가
from sklearn.metrics import mean_squared_error, r2_score

y_val_pred = model.predict(X_val)

# RMSE (pIC50 기준)
mse = mean_squared_error(y_val, y_val_pred)
rmse = mse ** 0.5

# R² Score
r2 = r2_score(y_val, y_val_pred)

print(f"📊 내부 검증 성능 (pIC50 기준)")
print(f"🔹 RMSE: {rmse:.4f}")
print(f"🔹 R²: {r2:.4f}")

✅ 제출용 파일 생성 완료 → rf_submission.csv
📊 내부 검증 성능 (pIC50 기준)
🔹 RMSE: 0.4171
🔹 R²: 0.9842


/var/folders/2q/v_79vqld72sf48ghq2z01j2m0000gn/T/ipykernel_68759/179155172.py:77: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  submission['ASK1_IC50_nM'].fillna(pIC50_to_IC50(y.mean()), inplace=True)
